<a href="https://colab.research.google.com/github/HaaseSchuetz/llm-evaluation-suite/blob/main/notebooks/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Evaluation Suite Demo

**Evaluate LLMs on standard benchmarks in Colab!**

 **No GPU required** (but recommended for faster evaluation).
 **Estimated Time:** ~10–30 mins (depending on benchmarks).
 **Storage:** ~5GB (for model weights).

---## What You’ll Do
1. Install dependencies.
2. Evaluate **Mistral-7B** on **MMLU** and **TruthfulQA**.
3. Generate a **comparison report**.
4. (Optional) Compare with **GPT-3.5-Turbo** (requires OpenAI API key).

---## Setup
Click **Runtime → Run all** to execute the entire notebook.

## 1 Install Dependencies

In [ ]:
# Install required packages
!pip install -q \
    torch==2.1.2 \
    transformers==4.38.2 \
    peft==0.8.2 \
    accelerate==0.25.0 \
    bitsandbytes==0.41.3 \
    datasets==2.16.1 \
    lm-evaluation-harness==0.4.0 \
    pandas \
    matplotlib \
    seaborn \
    tqdm

# Restart kernel (Colab-specific)
import os
if 'COLAB_GPU' in os.environ:
    print(' Restarting kernel to apply installed packages...')
    os.kill(os.getpid(), 9)

## 2 Imports and Setup

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from evaluation.evaluator import LLEvaluator

# Clone the repo (if not already)
!git clone https://github.com/HaaseSchuetz/llm-evaluation-suite.git 2>/dev/null || echo "Repo already cloned."
%cd llm-evaluation-suite

# Add to path
sys.path.append(str(Path.cwd()))

# Check GPU
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f'🔍 Using device: **{device}**')
if device == "cuda":
    print(f' GPU: {torch.cuda.get_device_name(0)}')

## 3 Initialize Evaluator

In [ ]:
# Initialize evaluator
evaluator = LLEvaluator(
    benchmarks_config="config/benchmarks.json",
    models_config="config/models.json",
    results_dir="colab_results"
)

## 4 Evaluate Mistral-7B on MMLU

In [ ]:
# Evaluate on MMLU (limit to 100 examples for speed)
print(' Evaluating Mistral-7B on MMLU...')
result_mmlu = evaluator.evaluate_model(
    model_name='mistral-7b',
    benchmark_name='mmlu',
    limit=100  # Remove for full evaluation
)

print(f' **MMLU Accuracy**: {result_mmlu.metrics["accuracy"]:.4f}')

## 5 Evaluate Mistral-7B on TruthfulQA

In [ ]:
print(' Evaluating Mistral-7B on TruthfulQA...')
result_tqa = evaluator.evaluate_model(
    model_name='mistral-7b',
    benchmark_name='truthfulqa',
    limit=100
)

print(f' **TruthfulQA Accuracy**: {result_tqa.metrics["accuracy"]:.4f}')

## 6 Compare with GPT-3.5-Turbo (Optional)
Set your OpenAI API key below to enable this.

In [ ]:
# Uncomment and set your API key
# import os
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# Evaluate GPT-3.5 on MMLU
try:
    print(' Evaluating GPT-3.5-Turbo on MMLU...')
    result_gpt_mmlu = evaluator.evaluate_model(
        model_name='gpt-3.5-turbo',
        benchmark_name='mmlu',
        limit=100
    )
    print(f' **GPT-3.5 MMLU Accuracy**: {result_gpt_mmlu.metrics["accuracy"]:.4f}')
except Exception as e:
    print(f' Skipping GPT-3.5 (API key not set): {str(e)}')
    result_gpt_mmlu = None

## 7 Generate Comparison Report

In [ ]:
# Collect results
results = [result_mmlu, result_tqa]
if 'result_gpt_mmlu' in locals() and result_gpt_mmlu:
    results.append(result_gpt_mmlu)

# Generate report
report = evaluator.generate_report(results, output_format='markdown')
print('\n **Evaluation Report:**\n')
print(report)

# Save report
evaluator.save_report(report, 'colab_demo', 'markdown')

## 8 Visualize Results

In [ ]:
# Convert results to DataFrame for plotting
data = []
for result in results:
    for metric, value in result.metrics.items():
        data.append({
            'Model': result.model,
            'Benchmark': result.benchmark,
            'Metric': metric,
            'Value': value
        })

df = pd.DataFrame(data)

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(
    data=df,
    x='Benchmark',
    y='Value',
    hue='Model',
    palette='viridis'
)
plt.title('LLM Evaluation Results')
plt.ylim(0, 1)
plt.ylabel('Accuracy')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Next Steps
### To Run Locally:
1. Clone the repo: `git clone https://github.com/HaaseSchuetz/llm-evaluation-suite.git`
2. Install dependencies: `pip install -r requirements.txt`
3. Run evaluations: `python scripts/evaluate.py --model mistral-7b --benchmark mmlu`

### To Extend:
- **Add more benchmarks** by editing `config/benchmarks.json`.
- **Compare more models** using `scripts/compare.py`.
- **Add custom metrics** in `evaluation/metrics/`.

---
** Star the [repo](https://github.com/HaaseSchuetz/llm-evaluation-suite) if you found this useful!**